## 调用工具的三种核心方式

* 手动绑定（`bing_tools`） \
* 高级智能体（`crate_agent`） \
* 底层图执行（`LangGraph`+`ToolNode`）

In [4]:
from langchain_core.messages import HumanMessage
# 示例1：最小示（需记下来）
from langchain_core.tools import tool
from langchain.agents import create_agent
from llm.my_llm import model
import dotenv
from rich import print as rprint

dotenv.load_dotenv(dotenv_path='../.env')
# 获取llm模型
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
# os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# model = ChatOpenAI(
#     model='MiniMax-M2.5',
#     # model='MiniMax-M2.5',
#     temperature=0.6
# )


# @tool
def get_weather(city: str) -> str:
    """查询不同城市的天气情况"""
    # print('+++++')
    return f"{city}是下雪"


agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt='你是一个天气助手'
)
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "帮我查询北京的天气"}]
# })
result = agent.invoke({
    "messages": [HumanMessage('帮我查询北京的天气')]
})
rprint(result)
print(result['messages'][-1].content)


{
    'messages': [
        HumanMessage(
            content='帮我查询北京的天气',
            additional_kwargs={},
            response_metadata={},
            id='11c0fa77-9edb-4659-add1-eaf411d271f0'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1413,
                    'prompt_tokens': 3132,
                    'total_tokens': 4545,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 1383,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 1413
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 3132}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-3a5a5183-9038-9585-8e6d-4615ff0ee831',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3671-9c30-7c63-862a-2782a38bf4b0-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_aafd8bfba4fe4f6f8a1c7b4f',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 3132,
                'output_tokens': 1413,
                'total_tokens': 4545,
                'input_token_details': {},
                'output_token_details': {'reasoning': 1383}
            }
        ),
        ToolMessage(
            content='北京是下雪',
            name='get_weather',
            id='a004f4bb-e49f-4aef-a1b7-c05cf0706eab',
            tool_call_id='call_aafd8bfba4fe4f6f8a1c7b4f'
        ),
        AIMessage(
            content='根据最新的信息，北京今天下雪了。\n\n雪景让城市披上了银装，虽然带来了寒冷和通勤的不便（如路面湿
滑、交通拥堵），但也为城市增添了一份独特的浪漫和美景。气象部门已发布道路结冰黄色和持续低温黄色预警，请注意防寒防冻
和交通安全。\n\n如果您计划出行，请做好保暖措施并预留更多的通勤时间。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 83,
                    'prompt_tokens': 1555,
                    'total_tokens': 1638,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 83
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 1555}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-9ce9410d-1ab3-98ff-a23c-46d8f6ca7788',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f3671-cae1-7490-b185-ec227f56918f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1555,
                'output_tokens': 83,
                'total_tokens': 1638,
                'input_token_details': {},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

根据最新的信息，北京今天下雪了。

雪景让城市披上了银装，虽然带来了寒冷和通勤的不便（如路面湿滑、交通拥堵），但也为城市增添了一份独特的浪漫和美景。气象部门已发布道路结冰黄色和持续低温黄色预警，请注意防寒防冻和交通安全。

如果您计划出行，请做好保暖措施并预留更多的通勤时间。


In [2]:
# 多伦对话，嵌入记忆
from langchain.agents import create_agent
from llm.my_llm import model
from langgraph.checkpoint.memory import MemorySaver

from rich import print as rprint
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

# 这会把 .env 里的键值对注入到 os.environ 中
load_dotenv()
# 使用工具
# os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
# 获取前三条网页记录
search = TavilySearch(max_results=3)
# print( search)
tools = [search]

memory = MemorySaver()

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=memory,  #关键！让 agent 记住历史
)
config = {"configurable": {"thread_id": "conversation-001"}}

'''
config=config 的用处就是让 Agent 知道你要操作哪个会话，
从而利用 checkpointer 实现跨轮次的对话记忆。没有它，每次调用都是全新的开始。
'''
# 第一轮
agent.invoke({
    "messages": [{"role": "user", "content": "今日北京的天气怎么样？"}]
}, config=config)

# 第二轮
res = agent.invoke({
    "messages": [{"role": "user", "content": "上海呢？"}]
}, config=config)

rprint(res)



{
    'messages': [
        HumanMessage(
            content='今日北京的天气怎么样？',
            additional_kwargs={},
            response_metadata={},
            id='dc87ac53-151e-42d0-8f8d-19777defd1d9'
        ),
        AIMessage(
            content='根据您提供的知识库数据，由于资料中包含多个时间节点的快照（主要集中在2026年6月底至7月初），北京
“今日”的天气需结合具体日期参考。综合最新记录，情况如下：\n\n📅 **若参考2026年6月30日的数据**\n- 
**天气现象**：雷阵雨\n- **气温**：24℃ ~ 32℃\n- **风力风向**：东北风 1级\n- **湿度**：81%\n- **空气质量**：优\n\n📅 
**若参考2026年7月6日（星期一）的数据**\n- **实况气温**：早晨约 25℃，午后最高温预计达 32℃\n- 
**风力风向**：西南风为主（<3级）\n- **相对湿度**：64%\n- 
**天气趋势**：受东北冷涡影响，近期云量较多，雷阵雨较为频繁；双休日预计有全市性雷雨过程，整体呈高温闷热态势。\n\n🌤️ 
**综合建议**\n此时段北京已进入夏季汛期，天气多变、降水集中且湿度较大。建议外出随身携带雨具，注意防范局地强对流天气
；同时因气温较高（普遍在30℃以上），请注意防暑降温与补水。\n\n*(注：以上信息均严格基于您提供的知识库文本整理，实际出
行请以当地气象台实时发布的最新预警为准。)*',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 3179,
                    'prompt_tokens': 5350,
                    'total_tokens': 8529,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 2870,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 3179
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 5350}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-ead035dc-9fed-92d7-91ed-f47f2e313e84',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f36e0-cd29-7501-88a3-665bcc580846-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 5350,
                'output_tokens': 3179,
                'total_tokens': 8529,
                'input_token_details': {},
                'output_token_details': {'reasoning': 2870}
            }
        ),
        HumanMessage(
            content='上海呢？',
            additional_kwargs={},
            response_metadata={},
            id='4de6061b-ab6d-4179-b3bc-c1dc8e26c071'
        ),
        AIMessage(
            content='根据您提供的知识库信息，今日（**2026年7月6日，星期一**）上海的天气情况如下：\n\n### 🌤️ 
今日天气概况\n*   **天气现象**：**阴转多云**。\n*   **气温范围**：**26℃ ~ 34℃**。\n*   
**实时体感**：目前气温较高，实况显示在 **34℃ - 35.5℃** 之间。\n*   
**风力与湿度**：风力较小（1级左右，风向西南或东北），相对湿度约 **49% - 50%**，体感较为闷热。\n*   
**空气质量**：优（AQI 35）。\n\n### 🔮 未来几天趋势\n上海正处于“梅雨”时节，天气变化较快，后续几天的预报显示：\n*   
**明日（7月7日）**：阴转小雨，气温 27℃ - 34℃。\n*   **后天（7月8日）**：小雨转阴，气温 27℃ - 33℃。\n*   
**周末（7月11日）**：预计有小雨，气温略有下降至 27℃ - 29℃。\n\n### 💡 出行建议\n*   
**防暑降温**：虽然云量较多，但最高气温仍可达34℃以上，且湿度较大，容易感到闷热，请注意防暑补水。\n*   
**防雨准备**：明后天及周末可能有雨水过程，出门建议随身携带雨具。\n*   
**关注预警**：近期新闻提示有“局部大雨到暴雨”风险，请留意气象部门发布的最新预警信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 955,
                    'prompt_tokens': 3844,
                    'total_tokens': 4799,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 597,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 955
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 3844}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-6b099f26-cf7e-9202-a838-37c5c4c2c66c',
                'finish_reason': 'stop',
                'logprobs': None
            },

In [45]:
# 流式输出
for chunk in agent.stream(
        {"messages": [{"role": "user", "content": "南京呢"}]},
        config=config,
        # 想调试 Agent 行为（看它用了哪些工具、中间结果对不对）→ 用 updates
        # 想给用户看流畅的打字效果 → 用 messages
        # 既想看过程又想实时输出 → 可以同时用两种模式（传 stream_mode=["updates", "messages"]）
        stream_mode="messages"  # 或 "messages" 看具体需求
):
    # print(chunk)
    print(chunk[0].content, end="", flush=True)



[{"title": "南京 - 中国气象局-天气预报-城市预报", "url": "https://weather.cma.cn/web/weather/58238.html", "content": "|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 12.5℃ | 14.8℃ | 15.1℃ | 14.8℃ | 14.6℃ | 14.5℃ | 14.5℃ | 14.4℃ |\n| 降水 | 无降水 | 4.9mm | 无降水 | 4.9mm | 0.1mm | 24.6mm | 0.1mm | 24.6mm |\n| 风速 | 7.9m/s | 7.4m/s | 7.9m/s | 7.6m/s | 6.4m/s | 3.2m/s | 1.9m/s | 1.3m/s |\n| 风向 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 |\n| 气压 | 1011.6hPa | 1009.7hPa | 1007.7hPa | 1006.9hPa | 1006.2hPa | 1004.7hPa | 1003.2hPa | 1003.6hPa |\n| 湿度 | 70.6% | 64.7% | 64.6% | 64.5% | 76.5% | 93.3% | 97.5% | 92.4% |\n| 云量 | 80% | 70% | 61.6% | 78.2% | 94.8% | 95.7% | 96.6% | 97.7% | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 12.6℃ | 18.7℃ | 19.3℃ | 18.5℃ | 15.5℃ | 12.3℃ | 12.1℃ | 11.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水

In [51]:
# 结构化输出
from pydantic import BaseModel


'''
    需要注意模型输出的信息中有这些字段才行
'''
# 结构化模型
class WeatherReport(BaseModel):
    city:str
    temperature:float
    condition:str

agent=create_agent(
    model=model,
    tools=tools,
    response_format=WeatherReport, # 强制结构化返回
    system_prompt='你是一个联网的AI助手'
)
result=agent.invoke({"messages":[{"role":"user","content":"安徽呢"}]})
weather = result.get("structured_response")  # WeatherReport 对象
print(weather.city, weather.temperature)


{'messages': [HumanMessage(content='安徽呢', additional_kwargs={}, response_metadata={}, id='32e596f2-2e08-46f7-bbac-282f2a033fe2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 281, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 66, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 130}}, 'model_provider': 'openai', 'model_name': 'MiniMax-M2.5', 'system_fingerprint': None, 'id': 'chatcmpl-55a78bdf-18a7-9804-adff-13c73a4cdc17', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d62bf-87b4-7842-86db-b1027ac65090-0', tool_calls=[{'name': 'WeatherReport', 'args': {'city': '安徽'}, 'id': 'call_b2f41207dd064022a13729ea', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 94, 'total_tokens': 375, 'input_token_deta

In [ ]:
# 示例2:旧版本

from langchain_core.tools import Tool
import os
import dotenv
from langchain_community.tools import TavilySearchResults, BingSearchResults
from langchain_openai import ChatOpenAI

dotenv.load_dotenv(dotenv_path='../.env')

# 获取Tavily搜索工具的实例
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
search = TavilySearchResults(max_results=3)

# 获取一个搜索的工具
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息"
)

# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='glm-5',
    temperature=0.6
)

# 通过create_agent调用invoke()，并得到响应


# 处理响应数据





In [1]:
# 示例3:使用create_agent
from langchain.agents import create_agent

import os
import dotenv
from langchain_community.tools import TavilySearchResults

from langchain_openai import ChatOpenAI

dotenv.load_dotenv(dotenv_path='../.env')
# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='glm-5',
    temperature=0.6
)
# 获取Tavily搜索工具的实例
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取前三条网页记录
search = TavilySearchResults(max_results=3)

tools = [search]
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="你是一个搜索AI助手"
)
# 3. 运行 Agent
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "查询一下今日的实时热点"}
    ]
})

# print(result)
print(result["messages"][-1].content)


/var/folders/sd/9y2p8s351dq2_t94lrs1040c0000gn/T/ipykernel_20147/2726070756.py:22: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=3)


PermissionDeniedError: Error code: 403 - {'error': {'message': 'The free tier of the model has been exhausted. If you wish to continue access the model on a paid basis, please disable the "use free tier only" mode in the management console.', 'type': 'AllocationQuota.FreeTierOnly', 'param': None, 'code': 'AllocationQuota.FreeTierOnly'}, 'id': 'chatcmpl-0697ad4b-72af-9d43-afd0-4242e21078ce', 'request_id': '0697ad4b-72af-9d43-afd0-4242e21078ce'}

In [5]:
# 多工具之间调用
from langchain.agents import create_agent
from langchain_core.tools import tool
from llm.my_llm import model
from dotenv import load_dotenv
from rich import print as rprint
from langchain_tavily import TavilySearch
load_dotenv()


# 分析工具
@tool
def analyze_stock_change(company_name: str, price_data: dict) -> str:
    """分析公司股价从年初至今的涨跌百分比。

    Args:
        company_name: 公司名称
        price_data: 包含 'current_price' 和 'start_price' 的字典
    """
    current_price = price_data['current_price']
    start_price = price_data['start_price']

    print(f"start_price==={start_price}")
    change_percent = ((current_price - start_price) / start_price) * 100
    return f"{company_name}股价从年初至今上涨/下跌了{change_percent:.2f}%"


# 获取前三条网页记录
search = TavilySearch(max_results=3)

tools = [search, analyze_stock_change]
agent = create_agent(
    model=model,
    tools=tools,
)
# 执行任务
res = agent.invoke({
    "messages": [{"role": "user", "content": "当前是2026年7月6日，宁德时代（300750）当前股价是多少？今年上涨了百分之几？"}]
})
rprint(res)



start_price===345.8


{
    'messages': [
        HumanMessage(
            content='当前是2026年7月6日，宁德时代（300750）当前股价是多少？今年上涨了百分之几？',
            additional_kwargs={},
            response_metadata={},
            id='9bcbea9c-0c44-41bf-9880-23eda13a199a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1254,
                    'prompt_tokens': 6294,
                    'total_tokens': 7548,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 1178,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 1254
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 6294}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.6-flash-2026-04-16',
                'system_fingerprint': None,
                'id': 'chatcmpl-fe372f29-42c7-9bf4-bb1e-fe578f459c87',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f36d3-6c2d-7812-b9f8-1525707135da-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {
                        'query': '宁德时代 300750 2026年年初股价 或 年初至今涨跌幅',
                        'time_range': 'year',
                        'search_depth': 'basic'
                    },
                    'id': 'call_5b8591ed37f5496b9be8d54b',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 6294,
                'output_tokens': 1254,
                'total_tokens': 7548,
                'input_token_details': {},
                'output_token_details': {'reasoning': 1178}
            }
        ),
        ToolMessage(
            content='{"query": "宁德时代 300750 2026年年初股价 或 年初至今涨跌幅", "follow_up_questions": null, 
"answer": null, "images": [], "results": [{"url": 
"https://www.futunn.com/quote/cn/stock-list/all-cn-stocks/top-market-cap?chain_id=Name1K9-3FXPhg.1km5q00&global_con
tent=%7B%22promote_id%22%3A13766%2C%22sub_promote_id%22%3A33%2C%22f%22%3A%22news.futunn.com%2Fpost%2F42961592%2Fctr
ip-group-s-9961-hk-looking-at-ota-overseas-market%22%7D", "title": "市场股票列表-股票行情中心-富途牛牛", "content":
"年初至今涨跌幅… 宁德时代300750 390.06 -2.95 -0.75% 2616.00亿 +6.15% +3.34%", "score": 0.8646883, "raw_content": 
null}, {"url": "https://www.moomoo.com/hans/quote/cn/stock-list/all-cn-stocks/top-market-cap", "title": "市值榜 - 
市场股票列表-股票行情中心-moomoo", "content": "指数最新行情,走势,及股票个股的实时报价。 宁德时代300750 380.00 -2.35
-0.61% 最新价涨跌额涨跌幅成交量… 幅年初至今涨跌幅", "score": 0.85216236, "raw_content": null}, {"url": 
"http://www.csrc.gov.cn/shanxi/c106408/c7614403/content.shtml", "title": "开年资本市场严监管“零容忍”信号持续释放 - 
中国证监会", "content": "## 开年资本市场严监管“零容忍”信号持续释放. 日期：2026-02-09 
\xa0\xa0\xa0\xa0来源：经济参考报. ###### 分享到：. 
随着证监会及各地证监局2026年“1号罚单”的相继落地，资本市场严监管“零容忍”的信号持续释放。同花顺iFinD数据显示，年初至
今，A股市场已有13家公司披露涉及立案调查的公告，其中证监会立案9起，其余为省级监察委员会立案，涉及6家上市公司、6位“关
键少数”。此外，证监会系统累计发出逾百份行政处罚及行政监管措施，涉及60余家上市公司，其中不乏市场高度关注的典型事件。
. 
从事由分布来看，涉嫌信息披露违法违规的占比最高，市场交易违法违规亦不容忽视。值得关注的是，监管层持续强化对上市公司“
关键少数”的个人责任追究，凸显出精准追责、穿透监管的鲜明导向。. 
开年首月，已有6家上市公司因涉嫌信息披露违法违规被立案，分别为宝馨科技（002514.SZ）、荃银高科（300087.SZ）、容百科技
（688005.SH）、向日葵（300111.SZ）、天普股份（605255.SH）和铂力特（688333.SH）。从具体事由来看，容百科技、向日葵、
天普股份均触及信息披露的真实性、准确性和完整性问题，反映出监管部门对此类行为持续保持高压态势。. 
1月19日，证监会称，容百科技1月14日披露日常经营重大合同的公告涉嫌误导性陈述，已对其立案调查，下一步将在全面调查的基
础上依法处理，切实维护市场健康稳定发展。. 
上述重大合同直指公司披露的与宁德时代（300750.SZ）签订的一份《磷酸铁锂正极材料采购合作协议》，总销售金额预计超1200亿
元。. 
公告当日，交易所即下发问询函，对信息披露准确性、协议具体内容、内幕交易防控等提出关切。在公司回复问询函的当日，证监
会对其立案调查。值得一提的是，2025年下半年，容百科技股价区间涨跌幅超90%，上述重大合同披露前的近一个月时间里股价区间
涨跌幅超35%。. 
药企向日葵的立案则与其一项备受关注的“跨界半导体”重组计划有关。证监会指出，近期，向日葵的重

### 多工具串联调用

In [3]:
import json
import httpx
from llm.my_llm import model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent



@tool
def get_weather(loc: str) -> str:
    """
    查询即时天气函数。
    :param loc: 必要参数，字符串类型，表示查询天气的具体城市英文名称，
                例如北京用 'Beijing'，上海用 'Shanghai'。
    :return: OpenWeather API 返回的 JSON 格式天气信息字符串。
    """
    url=f"https://wttr.in/{loc}?format=j1"
    response=httpx.get(url)
    data=response.json()
    return json.dumps(data)




# 定义工具列表
tools = [get_weather]

# 使用 LangGraph 预构建的 ReAct Agent，一步完成 agent 创建
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="你是天气助手，请根据用户的问题，给出相应的天气信息。"
)

# 直接调用 agent，输入自然语言问题
result = agent.invoke({"messages": [("user", "请问今天北京和上海的天气怎么样，哪个城市更热？")]})

# 输出结果（最后一条消息是助手的最终回复）
print(result["messages"][-1].content)

以下是今天（2026年6月22日）北京和上海的天气情况：

---

## ☀️ 北京

| 项目 | 数据 |
|------|------|
| **当前温度** | 27°C |
| **体感温度** | 27°C |
| **天气状况** | 晴朗 ☀️ |
| **最高温 / 最低温** | 33°C / 20°C |
| **湿度** | 54% |
| **风向风速** | 南偏西风 15km/h |
| **紫外线指数** | 7（较强） |
| **能见度** | 10km |

> 📌 北京今天以晴天为主，午后最高温可达 **33°C**，紫外线较强，注意防晒。晚间可能有零星小雨。

---

## 🌧️ 上海

| 项目 | 数据 |
|------|------|
| **当前温度** | 23°C |
| **体感温度** | 25°C |
| **天气状况** | 薄雾 / 小雨 🌧️ |
| **最高温 / 最低温** | 24°C / 22°C |
| **湿度** | 100% |
| **风向风速** | 东风 8km/h |
| **紫外线指数** | 2（较弱） |
| **能见度** | 4km |

> 📌 上海今天全天阴雨，以小雨和薄雾为主，湿度极高，能见度较低，出行请携带雨具。

---

## 🌡️ 哪个城市更热？

**北京明显更热！** 🥵

- 北京当前温度 **27°C**，最高可达 **33°C**
- 上海当前温度 **23°C**，最高仅 **24°C**

两地温差相差约 **9°C**（最高温对比），北京今天是一个典型的炎热晴天，而上海则凉爽潮湿，伴有降雨。
